# Диагностика prompt tuning: линейный зонд на замороженной xlm-roberta-base

Ни подсказки, ни LoRA здесь нет. Берём замороженную `xlm-roberta-base`, получаем представления текстов и обучаем только логистическую регрессию.

Вопрос: есть ли в замороженной модели сигнал, по которому отличаются атаки от безопасных текстов.
- Если зонд даёт val F1 заметно выше 0,67 (это F1 константного ответа «атака»), сигнал есть, и проблема в подсказке.
- Если нет, проблема в самой задаче или данных, и подсказка тут ни при чём.

**golden сюда не загружать.** Диагностика идёт только по val.
Runtime -> Change runtime type -> **T4 GPU**. Запускайте ячейки сверху вниз, в ячейке 2 загрузите `train.jsonl` и `val.jsonl`.

In [ ]:
!pip install -q transformers scikit-learn
import torch
assert torch.cuda.is_available(), "включите GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()  # выберите train.jsonl и val.jsonl
for name in ("train.jsonl", "val.jsonl"):
    assert name in uploaded, f"не загружен {name}"
assert "golden.jsonl" not in uploaded, "golden сюда не загружаем"

In [ ]:
import json, numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL, MAX_LEN = "xlm-roberta-base", 384
tok = AutoTokenizer.from_pretrained(MODEL)
enc_model = AutoModel.from_pretrained(MODEL).cuda().eval().half()

def load(name):
    rows = [json.loads(l) for l in uploaded[name].decode("utf-8").splitlines() if l.strip()]
    return [r["text"] for r in rows], np.array([int(r["label"] == "attack") for r in rows])

@torch.no_grad()
def features(texts, batch=64):
    cls, mean = [], []
    for i in range(0, len(texts), batch):
        e = tok(texts[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
        h = enc_model(**e).last_hidden_state.float()
        m = e["attention_mask"].unsqueeze(-1).float()
        cls.append(h[:, 0].cpu().numpy())
        mean.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return {"cls": np.concatenate(cls), "mean": np.concatenate(mean)}

tr_texts, y_tr = load("train.jsonl")
va_texts, y_va = load("val.jsonl")
F_tr, F_va = features(tr_texts), features(va_texts)
print({k: v.shape for k, v in F_tr.items()}, "атак в val:", round(y_va.mean(), 3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

# C фиксирован заранее (1.0), на val не подбирается: это диагностика, а не выбор модели
for pooling in ("cls", "mean"):
    sc = StandardScaler().fit(F_tr[pooling])
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(F_tr[pooling]), y_tr)
    p = clf.predict_proba(sc.transform(F_va[pooling]))[:, 1]
    best = max((f1_score(y_va, p >= t), t) for t in np.linspace(0.05, 0.95, 19))
    print(f"{pooling}: val ROC AUC {roc_auc_score(y_va, p):.3f}, F1 при 0.5 {f1_score(y_va, p >= 0.5):.3f}, "
          f"лучший F1 {best[0]:.3f} при пороге {best[1]:.2f}")
print("опорная точка: константный ответ «атака» даёт F1", round(f1_score(y_va, np.ones_like(y_va)), 3))

Как читать результат:
- **AUC около 0,9 и выше** — в замороженной модели есть сигнал. Тогда prompt tuning не находит его из-за настроек, и стоит пробовать другую инициализацию или отдельную скорость для головы.
- **AUC около 0,7** — сигнал слабый. Prompt tuning на такой базе, скорее всего, безнадёжен, и тогда результат честно записывается как отрицательный.
- **AUC около 0,5** — признаки вообще не разделяют классы. Это значит, что данные или разметка проблемные, и сравнивать методы на них бессмысленно. Тогда LoRA показала хорошее качество по другой причине, которую надо проверять.